# CMPE 258 Team 7: hardened benchmark runs (Colab T4)

**Robust Neural Image Watermarking with Failure-Guided Attack Sampling (FGAS)**

Official numbers for the hardened benchmark come from this notebook on a **Colab T4 GPU**. Two teammates run it on their own Colab accounts at the same time, each with a different `MY_JOBS` list (section 3). Every run writes straight to the team Drive folder, so a Colab disconnect loses at most one epoch: re-running the job cell resumes from `last.pt`.

**Before running:** `Runtime -> Change runtime type -> T4 GPU`, set `MY_JOBS`, then `Runtime -> Run all`.

What it does:
1. Clones the repo, installs dependencies, runs the unit tests.
2. Data: COCO val2017 (4500 train / 500 val) for training and model selection; DIV2K valid HR (100 images) as the unseen test set. DIV2K train is only needed for the continuation experiment (D). Zips are cached on Drive.
3. Runs the selected jobs: calibration (`calib_*`), continuation from the DIV2K baseline (`*_continue_s*`), and the main from-scratch comparison (`hard_uniform_s*` vs `hard_fgas_s*`).
4. Severity sweep on the DIV2K test set, multi-seed aggregation, and the success-criterion check.
5. Collects the small result files into `results/checkin3/` and copies them to Drive for committing.

In [1]:
# Fail fast on a CPU runtime: Runtime -> Change runtime type -> T4 GPU, then Runtime -> Run all.
import torch
if not torch.cuda.is_available():
    raise SystemExit('No GPU attached. Switch to Runtime -> Change runtime type -> T4 GPU and run again.')
print('GPU:', torch.cuda.get_device_name(0))
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv

GPU: Tesla T4
name, memory.total [MiB], driver_version
Tesla T4, 15360 MiB, 580.82.07


In [2]:
# Team Drive folder: dataset zip cache, run folders (checkpoints, logs) and collected results.
# Change DRIVE_DIR if the shared team folder lives somewhere else in your Drive.
from google.colab import drive
drive.mount('/content/drive')
DRIVE_DIR = '/content/drive/MyDrive/CMPE258_Team7'
RUNS_DIR = f'{DRIVE_DIR}/runs_hardened'
import os
os.makedirs(f'{DRIVE_DIR}/cache', exist_ok=True)
os.makedirs(RUNS_DIR, exist_ok=True)

Mounted at /content/drive


In [3]:
REPO_URL = 'https://github.com/TaehunLeee5/CMPE258_Team7.git'
BRANCH = 'main'
%cd /content
!test -d CMPE258_Team7 || git clone -b {BRANCH} {REPO_URL}
%cd /content/CMPE258_Team7
!git pull --ff-only && git log -1 --oneline
!pip install -q -r requirements.txt
!python -m pytest -q

/content
Cloning into 'CMPE258_Team7'...
remote: Enumerating objects: 296, done.
remote: Counting objects: 100% (296/296), done.
remote: Compressing objects: 100% (179/179), done.
remote: Total 296 (delta 140), reused 237 (delta 95), pack-reused 0 (from 0)
Receiving objects: 100% (296/296), 6.87 MiB | 12.21 MiB/s, done.
Resolving deltas: 100% (140/140), done.
/content/CMPE258_Team7
Already up to date.
884ead2 (HEAD -> main, origin/main, origin/HEAD) Warm up on the four basic attacks before the hardened attack set
........................................................................ [ 75%]
........................                                                 [100%]
96 passed in 11.23s


## 1. Data
COCO val2017 is split by sorted filename (first 4500 train, last 500 val) and downscaled to a short side of 256, stored as PNG. DIV2K valid HR is the held-out test set: it is never used for training, checkpoint selection or FGAS.

In [4]:
!python scripts/download_coco.py --config configs/hard_base.yaml --cache-dir {DRIVE_DIR}/cache
!python scripts/prepare_coco.py --config configs/hard_base.yaml
!python scripts/validate_coco.py --config configs/hard_base.yaml

    777.8 / 777.8 MB
Extracting /content/drive/MyDrive/CMPE258_Team7/cache/val2017.zip -> data/COCO
val2017: 5000 images in data/COCO/val2017
5000 images, 0 dropped (short side < 128)
train: 100% 4500/4500 [03:19<00:00, 22.56it/s]
val: 100% 500/500 [00:22<00:00, 22.44it/s]
Prepared 4500 train, 500 val images in data/coco_prepared
Validated 4500 train / 500 val COCO images
Validation crops: 1000 (2 per image)
Saved 8 deterministic crop samples to results/coco_samples


In [5]:
# DIV2K: valid HR is the test set for every hardened run. Train HR is only needed for the
# continuation experiment (D), which starts from the committed DIV2K baseline checkpoint.
NEED_DIV2K_TRAIN = False
splits = 'train valid' if NEED_DIV2K_TRAIN else 'valid'
!python scripts/download_div2k.py --config configs/baseline.yaml --splits {splits} --cache-dir {DRIVE_DIR}/cache
if NEED_DIV2K_TRAIN:
    !python scripts/prepare_div2k.py --config configs/baseline.yaml
else:
    !python scripts/prepare_div2k.py --config configs/baseline.yaml --splits val

    428.2 / 428.2 MB
Extracting /content/drive/MyDrive/CMPE258_Team7/cache/DIV2K_valid_HR.zip -> data/DIV2K
valid: 100 images in data/DIV2K/DIV2K_valid_HR
val: 100% 100/100 [00:30<00:00,  3.24it/s]
Prepared 100 val images in data/div2k_prepared


## 2. Jobs
| Job | Config | What |
|---|---|---|
| `calib_w100`, `calib_w200`, `calib_w400` | `configs/calib_w*.yaml` | 20-epoch uniform runs with final image-loss weight 100 / 200 / 400 |
| `calib_w100_strong` | `configs/calib_w100_strong.yaml` | stronger noise / JPEG / crop, only if every attack stays >= 99% |
| `baseline_continue_s{42,43,44}`, `fgas_continue_s{42,43,44}` | `configs/baseline_continue.yaml`, `configs/fgas.yaml` | Experiment D: 10 more epochs from the DIV2K baseline, uniform vs FGAS |
| `hard_uniform_s{42,43,44}`, `hard_fgas_s{42,43,44}` | `configs/hard_uniform.yaml`, `configs/hard_fgas.yaml` | Experiment E: 60 epochs from scratch, uniform vs FGAS (image-loss weight set from calibration) |
| `hard_fgas_p2_s{42,43,44}` | `configs/hard_fgas_p2.yaml` | Pre-registered fallback (FGAS with power 2), only if E misses the success criterion |

Split the jobs between the two Colab accounts so nobody runs the same job twice. A job that already finished is skipped, and an interrupted one resumes from its `last.pt` on Drive.

In [6]:
BASELINE_CKPT = 'results/baseline_uniform/best.pt'
ALL_JOBS = {
    'calib_w400': ('configs/calib_w400.yaml', 42, []),
    'calib_w100': ('configs/calib_w100.yaml', 42, []),
    'calib_w200': ('configs/calib_w200.yaml', 42, []),
    'calib_w100_strong': ('configs/calib_w100_strong.yaml', 42, []),
}
for seed in (42, 43, 44):
    ALL_JOBS[f'baseline_continue_s{seed}'] = ('configs/baseline_continue.yaml', seed, [f'train.init_checkpoint={BASELINE_CKPT}'])
    ALL_JOBS[f'fgas_continue_s{seed}'] = ('configs/fgas.yaml', seed, [f'train.init_checkpoint={BASELINE_CKPT}'])
    ALL_JOBS[f'hard_uniform_s{seed}'] = ('configs/hard_uniform.yaml', seed, [])
    ALL_JOBS[f'hard_fgas_s{seed}'] = ('configs/hard_fgas.yaml', seed, [])
    ALL_JOBS[f'hard_fgas_p2_s{seed}'] = ('configs/hard_fgas_p2.yaml', seed, [])  # fallback, only if needed

# Edit per account, e.g. ['calib_w100', 'calib_w400'] on one and ['calib_w200', 'baseline_continue_s42', ...] on the other.=
MY_JOBS = ['hard_uniform_s42', 'hard_fgas_s42']

In [7]:
import csv, pathlib, sys
sys.path.insert(0, 'src')
from wm.config import load_config


def finished(name, config, overrides):
    metrics = pathlib.Path(RUNS_DIR) / name / 'metrics.csv'
    if not metrics.exists():
        return False
    epochs = load_config(config, overrides)['train']['epochs']
    rows = list(csv.DictReader(open(metrics)))
    return bool(rows) and int(rows[-1]['epoch']) >= epochs


def run_job(name):
    config, seed, extra = ALL_JOBS[name]
    overrides = [f'experiment.name={name}', f'experiment.seed={seed}', f'experiment.output_dir={RUNS_DIR}', *extra]
    if finished(name, config, overrides):
        print(f'{name}: already finished, skipping')
        return
    last = pathlib.Path(RUNS_DIR) / name / 'last.pt'
    if last.exists():
        overrides.append(f'train.resume={last}')
        print(f'{name}: resuming from {last}')
    sets = ' '.join(overrides)
    get_ipython().system(f'python scripts/train.py --config {config} --set {sets}')
    if not finished(name, config, overrides):
        raise RuntimeError(f'{name} did not finish; re-run this cell to resume it')


for job in MY_JOBS:
    run_job(job)

[epoch 1/60] loss=0.7022 psnr=23.09 ssim=0.798 bit_acc: clean=0.629 noise=0.629 blur=0.624 brightness=0.609 resize=0.628 jpeg=0.627 crop=0.585 occlusion=0.609 (38s)
[epoch 2/60] loss=0.3906 psnr=18.66 ssim=0.726 bit_acc: clean=0.979 noise=0.977 blur=0.978 brightness=0.975 resize=0.979 jpeg=0.978 crop=0.749 occlusion=0.940 (37s)
Train bit accuracy 0.996: image-loss ramp starts at epoch 4
[epoch 3/60] loss=0.0719 psnr=20.82 ssim=0.803 bit_acc: clean=1.000 noise=1.000 blur=1.000 brightness=0.997 resize=1.000 jpeg=1.000 crop=0.782 occlusion=0.978 (37s)
[epoch 4/60] loss=0.1933 psnr=27.48 ssim=0.934 bit_acc: clean=0.996 noise=0.993 blur=0.995 brightness=0.972 resize=0.996 jpeg=0.995 crop=0.827 occlusion=0.974 (37s)
[epoch 5/60] loss=0.1849 psnr=29.41 ssim=0.954 bit_acc: clean=0.998 noise=0.994 blur=0.997 brightness=0.980 resize=0.998 jpeg=0.997 crop=0.857 occlusion=0.979 (37s)
[epoch 6/60] loss=0.1672 psnr=30.79 ssim=0.964 bit_acc: clean=0.987 noise=0.977 blur=0.983 brightness=0.965 resize=

## 3. Severity sweep on the DIV2K test set
Every attack at 4-5 levels, from mild to beyond the training strength (`attacks.sweep` in `configs/hard_base.yaml`), for every finished main-comparison run.

In [8]:
import glob
for ckpt in sorted(glob.glob(f'{RUNS_DIR}/hard_*_s*/best.pt')):
    run_dir = os.path.dirname(ckpt)
    if not os.path.exists(f'{run_dir}/sweep.csv'):
        !python scripts/sweep.py --checkpoint {ckpt} --out {run_dir}/sweep.csv
sweeps = ' '.join(sorted(glob.glob(f'{RUNS_DIR}/hard_*_s*/sweep.csv')))
if sweeps:
    !python scripts/sweep.py --plot {sweeps} --out results/checkin3/figures/severity_sweep.png

Swept 29 attack levels on 400 test crops from /content/drive/MyDrive/CMPE258_Team7/runs_hardened/hard_fgas_s42/best.pt
  clean                                    bit_acc 0.9997  msg_acc 0.9925
  noise       0.05                         bit_acc 0.9983  msg_acc 0.9575
  noise       0.1                          bit_acc 0.9779  msg_acc 0.4925
  noise       0.15                         bit_acc 0.9126  msg_acc 0.0675
  noise       0.2                          bit_acc 0.8294  msg_acc 0.0025
  blur        1.0                          bit_acc 0.9988  msg_acc 0.9675
  blur        2.0                          bit_acc 0.9963  msg_acc 0.9250
  blur        3.0                          bit_acc 0.9902  msg_acc 0.8275
  blur        4.0                          bit_acc 0.9761  msg_acc 0.6900
  brightness  0.7-1.3                      bit_acc 0.9926  msg_acc 0.9600
  brightness  0.5-1.5                      bit_acc 0.9905  msg_acc 0.9225
  brightness  0.3-1.7                      bit_acc 0.9881  msg_acc 

## 4. Tables: calibration, continuation (D), main comparison (E)
Mean +- std over seeds. The criterion check compares FGAS with the matched uniform baseline on the DIV2K test set: worst-case attack bit accuracy +2 pp, PSNR within 0.5 dB, mean attacked accuracy not lower by more than 0.5 pp, clean accuracy >= 98%.

In [9]:
OUT = 'results/checkin3'
!python scripts/aggregate_seeds.py {RUNS_DIR}/calib_* --file best_eval.json --no-group --out {OUT}/calibration/summary
!python scripts/aggregate_seeds.py {RUNS_DIR}/baseline_continue_s* {RUNS_DIR}/fgas_continue_s* --file best_eval.json --out {OUT}/exp_d
!python scripts/aggregate_seeds.py {RUNS_DIR}/hard_uniform_s* {RUNS_DIR}/hard_fgas_s* --out {OUT}/main_comparison --criterion hard_uniform hard_fgas
if glob.glob(f'{RUNS_DIR}/hard_fgas_p2_s*'):
    !python scripts/aggregate_seeds.py {RUNS_DIR}/hard_uniform_s* {RUNS_DIR}/hard_fgas_p2_s* --out {OUT}/fallback_p2 --criterion hard_uniform hard_fgas_p2

skipping /content/drive/MyDrive/CMPE258_Team7/runs_hardened/calib_*: no best_eval.json
No best_eval.json found in the given run directories
skipping /content/drive/MyDrive/CMPE258_Team7/runs_hardened/baseline_continue_s*: no best_eval.json
skipping /content/drive/MyDrive/CMPE258_Team7/runs_hardened/fgas_continue_s*: no best_eval.json
No best_eval.json found in the given run directories
{
  "uniform": "hard_uniform",
  "fgas": "hard_fgas",
  "delta_worst_pp": 7.359375000000002,
  "delta_psnr_db": -0.34525848388671676,
  "delta_mean_attacked_pp": 0.8046874999999898,
  "fgas_clean_bit_acc": 0.9996875,
  "checks": {
    "worst_case_gain_ge_2pp": true,
    "psnr_within_0.5db": true,
    "mean_attacked_drop_le_0.5pp": true,
    "clean_ge_98": true
  },
  "met": true
}
| metric | hard_fgas (n=1) | hard_uniform (n=1) |
|---|---|---|
| clean_bit_acc | 99.97 ± 0.00 | 99.97 ± 0.00 |
| clean_msg_acc | 99.25 ± 0.00 | 99.00 ± 0.00 |
| noise_bit_acc | 97.87 ± 0.00 | 98.92 ± 0.00 |
| noise_msg_acc | 5

## 5. Collect result files and copy them to Drive
Small files only: metrics, configs, environment info, evaluation JSONs, FGAS history, sweeps, and one `best.pt` per method (seed 42). Whoever commits downloads `results/checkin3/` from Drive into the repo.

In [10]:
import shutil
KEEP = ['metrics.csv', 'config.yaml', 'run_info.json', 'best_eval.json', 'best_test_eval.json', 'fgas_history.json', 'sweep.csv']
for run_dir in sorted(glob.glob(f'{RUNS_DIR}/*')):
    name = os.path.basename(run_dir)
    dest = pathlib.Path(OUT) / 'runs' / name
    dest.mkdir(parents=True, exist_ok=True)
    for f in KEEP:
        if os.path.exists(f'{run_dir}/{f}'):
            shutil.copy(f'{run_dir}/{f}', dest / f)
    if name.endswith('_s42') and os.path.exists(f'{run_dir}/best.pt'):
        shutil.copy(f'{run_dir}/best.pt', dest / 'best.pt')
shutil.copytree(OUT, f'{DRIVE_DIR}/results_checkin3', dirs_exist_ok=True)
print('Copied', OUT, 'to', f'{DRIVE_DIR}/results_checkin3')
!find {OUT} -maxdepth 2 | sort | head -50

Copied results/checkin3 to /content/drive/MyDrive/CMPE258_Team7/results_checkin3
results/checkin3
results/checkin3/exp_d.csv
results/checkin3/exp_d.md
results/checkin3/figures
results/checkin3/figures/severity_sweep.png
results/checkin3/main_comparison_criterion.json
results/checkin3/main_comparison.csv
results/checkin3/main_comparison.md
results/checkin3/runs
results/checkin3/runs/baseline_continue_s42
results/checkin3/runs/baseline_continue_s43
results/checkin3/runs/baseline_continue_s44
results/checkin3/runs/fgas_continue_s42
results/checkin3/runs/fgas_continue_s43
results/checkin3/runs/fgas_continue_s44
results/checkin3/runs/hard_fgas_s42
results/checkin3/runs/hard_uniform_s42
